In [1]:
import subprocess
import sys

# Install required packages
packages = ['statsmodels', 'scikit-learn']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All required packages installed successfully!")

✓ All required packages installed successfully!


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# --- STEP 1: LOAD DATA ---
FILE_NAME = 'energy_data_3_years.csv'
try:
    data = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"Error: The file '{FILE_NAME}' was not found.")
    exit()

# Prepare data
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date').reset_index(drop=True)
data = data.set_index('Date')

print("Data loaded successfully!")
print(f"Date range: {data.index.min().date()} to {data.index.max().date()}")
print(f"Total records: {len(data)}")
print(f"\nColumns: {list(data.columns)}")

# --- STEP 2: PREPARE SOLAR DATA ---
solar_col = 'Solar_Energy_Produced_kWh'
solar_data = data[solar_col].ffill().bfill()

print(f"\n{'='*70}")
print("SOLAR ENERGY - HISTORICAL STATISTICS")
print(f"{'='*70}")
print(f"Mean: {solar_data.mean():.2f} kWh")
print(f"Std: {solar_data.std():.2f} kWh")
print(f"Min: {solar_data.min():.2f} kWh")
print(f"Max: {solar_data.max():.2f} kWh")

# --- STEP 3: SEASONAL DECOMPOSITION ---
print(f"\n{'='*70}")
print("SEASONAL DECOMPOSITION")
print(f"{'='*70}")
decomposition = seasonal_decompose(solar_data, model='additive', period=365)
print("✓ Decomposition completed")
print(f"  Trend component: {decomposition.trend.notna().sum()} values")
print(f"  Seasonal component: {decomposition.seasonal.notna().sum()} values")
print(f"  Residual component: {decomposition.resid.notna().sum()} values")

# --- STEP 4: TRAIN SARIMA MODEL ---
print(f"\n{'='*70}")
print("TRAINING SARIMA MODEL")
print(f"{'='*70}")

# SARIMA parameters: (p,d,q) x (P,D,Q,s)
# p=1, d=1, q=1 (non-seasonal)
# P=1, D=1, Q=1, s=365 (seasonal with yearly period)
print("Training SARIMA(1,1,1)x(1,1,1,365)...")
sarima_model = SARIMAX(
    solar_data,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 365),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_fit = sarima_model.fit(disp=False)
print("✓ Model trained successfully!")
print(f"\nModel Summary:")
print(sarima_fit.summary())

# --- STEP 5: GENERATE 2026 FORECAST ---
print(f"\n{'='*70}")
print("GENERATING 2026 FORECAST (365 DAYS)")
print(f"{'='*70}")

# Forecast next 365 days
forecast_steps = 365
solar_forecast = sarima_fit.get_forecast(steps=forecast_steps)
solar_pred = solar_forecast.predicted_mean
solar_ci = solar_forecast.conf_int(alpha=0.05)

# Ensure positive values
solar_pred = np.maximum(solar_pred.values, 0)

# Create forecast dataframe
forecast_dates = pd.date_range(start='2026-01-01', periods=365, freq='D')
forecast_df = pd.DataFrame({
    'Date': forecast_dates,
    'Solar_Energy_Forecast_kWh': solar_pred,
    'Lower_CI': np.maximum(solar_ci.iloc[:, 0].values, 0),
    'Upper_CI': solar_ci.iloc[:, 1].values
})

print(f"✓ Forecast generated!")
print(f"\nForecast Statistics:")
print(f"Mean: {solar_pred.mean():.2f} kWh")
print(f"Std: {solar_pred.std():.2f} kWh")
print(f"Min: {solar_pred.min():.2f} kWh")
print(f"Max: {solar_pred.max():.2f} kWh")

# --- STEP 6: WIND ENERGY FORECAST ---
wind_col = 'Wind_Energy_Produced_kWh'
if wind_col in data.columns:
    print(f"\n{'='*70}")
    print("WIND ENERGY - SARIMA FORECAST")
    print(f"{'='*70}")
    
    wind_data = data[wind_col].ffill().bfill()
    print(f"Mean: {wind_data.mean():.2f} kWh")
    print(f"Std: {wind_data.std():.2f} kWh")
    
    # Train SARIMA for wind
    print("\nTraining SARIMA(1,1,1)x(1,1,1,365) for wind...")
    wind_model = SARIMAX(
        wind_data,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 365),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    wind_fit = wind_model.fit(disp=False)
    
    # Forecast wind
    wind_forecast = wind_fit.get_forecast(steps=365)
    wind_pred = np.maximum(wind_forecast.predicted_mean.values, 0)
    wind_ci = wind_forecast.conf_int(alpha=0.05)
    
    # Add to forecast dataframe
    forecast_df['Wind_Energy_Forecast_kWh'] = wind_pred
    forecast_df['Wind_Lower_CI'] = np.maximum(wind_ci.iloc[:, 0].values, 0)
    forecast_df['Wind_Upper_CI'] = wind_ci.iloc[:, 1].values
    
    print(f"✓ Wind forecast completed!")
    print(f"Mean: {wind_pred.mean():.2f} kWh")
    print(f"Max: {wind_pred.max():.2f} kWh")

# --- STEP 7: SAVE AND DISPLAY RESULTS ---
print(f"\n{'='*70}")
print("FORECAST RESULTS")
print(f"{'='*70}")

# Save forecast
output_file = 'sarima_forecast_2026.csv'
forecast_df.to_csv(output_file, index=False)
print(f"✓ Forecast saved to '{output_file}'")

# Display sample
print(f"\nForecast Sample (First 10 days):")
print(forecast_df.head(10).to_string(index=False))

# Monthly summary
forecast_df['Month'] = pd.to_datetime(forecast_df['Date']).dt.to_period('M')
monthly = forecast_df.groupby('Month')['Solar_Energy_Forecast_kWh'].agg(['mean', 'min', 'max', 'sum'])

print(f"\n{'='*70}")
print("MONTHLY SUMMARY (2026)")
print(f"{'='*70}")
print(monthly.to_string())

# Total energy
total_solar = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"\nTotal Solar Energy (2026): {total_solar:,.2f} kWh")

if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    total_wind = forecast_df['Wind_Energy_Forecast_kWh'].sum()
    total_combined = total_solar + total_wind
    print(f"Total Wind Energy (2026): {total_wind:,.2f} kWh")
    print(f"Combined Total (2026): {total_combined:,.2f} kWh")
    print(f"\nSolar: {(total_solar/total_combined)*100:.1f}%")
    print(f"Wind: {(total_wind/total_combined)*100:.1f}%")

print(f"\n{'='*70}")
print("✓ SARIMA FORECASTING COMPLETE")
print(f"{'='*70}")

Data loaded successfully!
Date range: 2022-12-10 to 2025-12-08
Total records: 1095

Columns: ['Avg_Solar_Irradiance_W_m2', 'Avg_Wind_Speed_m_s', 'Solar_Energy_Produced_kWh', 'Wind_Energy_Produced_kWh']

SOLAR ENERGY - HISTORICAL STATISTICS
Mean: 9.53 kWh
Std: 2.01 kWh
Min: 4.50 kWh
Max: 14.52 kWh

SEASONAL DECOMPOSITION
✓ Decomposition completed
  Trend component: 731 values
  Seasonal component: 1095 values
  Residual component: 731 values

TRAINING SARIMA MODEL
Training SARIMA(1,1,1)x(1,1,1,365)...


c:\Users\NANCY_SINGH\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\NANCY_SINGH\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
